In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
import numpy as np
from collections import Counter
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
from torchvision import transforms
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import torch.optim as optim
from torch import nn
import os
import zipfile
from pathlib import Path
import requests
!pip install torchinfo
from torchinfo import summary
from torch.utils.data import TensorDataset, DataLoader, Dataset
from tqdm.auto import tqdm
import gc
from PIL import Image
from torchvision.models import GoogLeNet_Weights

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)
print("Training labels shape:", y_train.shape)
print("Unique classes:", len(set(y_train)))

# Show first image
plt.imshow(x_train[0], cmap='gray')
plt.title(f"Label: {y_train[0]}")
plt.axis('off')
plt.show()

In [ ]:
class_counts = Counter(y_train)
print("Original class counts:", class_counts)

plt.bar(class_counts.keys(), class_counts.values())
plt.xlabel("Digit Class")
plt.ylabel("Count")
plt.title("Original Class Distribution (Before Augmentation)")
plt.show()

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=(0.8, 1.2),
    fill_mode='nearest'
)

In [ ]:
target_per_class = max(class_counts.values())

X_balanced = []
y_balanced = []

print("\n🔹 Creating balanced dataset...")

for digit in range(10):
    digit_images = x_train[y_train == digit]
    count = len(digit_images)

    if count < target_per_class:
        extra_needed = target_per_class - count
        print(f"Digit {digit}: needs {extra_needed} more images")

        digit_images = np.expand_dims(digit_images, -1)
        generator = datagen.flow(digit_images, batch_size=32, shuffle=True)

        new_images = []
        while len(new_images) < extra_needed:
            batch = next(generator)[0]
            new_images.append(batch.squeeze().astype(np.uint8))

        all_images = np.concatenate([digit_images.squeeze(), np.array(new_images)], axis=0)

    else:
        all_images = digit_images

    all_labels = np.full(len(all_images), digit)

    X_balanced.append(all_images)
    y_balanced.append(all_labels)

x_train_bal = np.concatenate(X_balanced)
y_train_bal = np.concatenate(y_balanced)

print("\n✅ Dataset balanced successfully!")
print("New training data shape:", x_train_bal.shape)
print("New labels shape:", y_train_bal.shape)


In [ ]:
new_counts = Counter(y_train_bal)
print("Balanced class counts:", new_counts)

plt.bar(new_counts.keys(), new_counts.values())
plt.xlabel("Digit Class")
plt.ylabel("Count")
plt.title("Balanced Class Distribution (After Augmentation)")
plt.show()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
manual_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
weights = torchvision.models.GoogLeNet_Weights.DEFAULT
weights

In [ ]:
auto_transforms = weights.transforms()
auto_transforms

In [ ]:
weights = torchvision.models.GoogLeNet_Weights.DEFAULT  # best pretrained weights
model = torchvision.models.googlenet(weights=weights).to(device)


In [ ]:
summary(model=model,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

# Frozen feature extractor

In [ ]:
for name, param in model.named_parameters():
    if not name.startswith('fc'):
        param.requires_grad = False


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

output_shape = len(set(y_train))

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=output_shape,
                    bias=True)).to(device)

In [ ]:
summary(model,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
x_train_tensor = torch.tensor(x_train_bal).unsqueeze(1).float() / 255.0  # (N, 1, 28, 28)
x_test_tensor = torch.tensor(x_test).unsqueeze(1).float() / 255.0
y_train_tensor = torch.tensor(y_train_bal).long()
y_test_tensor = torch.tensor(y_test).long()

# Convert grayscale to 3-channel (since GoogLeNet expects RGB)
x_train_tensor = x_train_tensor.repeat(1, 3, 1, 1)
x_test_tensor = x_test_tensor.repeat(1, 3, 1, 1)

# Apply transforms (resize + normalize)
train_data = []
for i in range(len(x_train_tensor)):
    img = transforms.ToPILImage()(x_train_tensor[i])
    img = manual_transforms(img)
    train_data.append(img)
x_train_tensor = torch.stack(train_data)

test_data = []
for i in range(len(x_test_tensor)):
    img = transforms.ToPILImage()(x_test_tensor[i])
    img = manual_transforms(img)
    test_data.append(img)
x_test_tensor = torch.stack(test_data)

# 2. Create DataLoaders
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Training loop
epochs = 7
for epoch in range(epochs):
    model.train()
    train_loss, correct = 0, 0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        X, y = X.to(device), y.to(device)

        # Forward pass
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # Backward + optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accuracy
        correct += (y_pred.argmax(1) == y).sum().item()

    train_acc = correct / len(train_dataset)
    print(f"Train loss: {train_loss/len(train_loader):.4f} | Train acc: {train_acc*100:.2f}%")

    # Evaluation loop
    model.eval()
    test_loss, test_correct = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            test_correct += (y_pred.argmax(1) == y).sum().item()
    test_acc = test_correct / len(test_dataset)
    print(f"Test loss: {test_loss/len(test_loader):.4f} | Test acc: {test_acc*100:.2f}%\n")


In [ ]:
# Lists to store results
train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []

# Training loop with metric tracking
epochs = 5  # or however many you want
for epoch in range(epochs):
    model.train()
    train_loss, correct = 0, 0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        correct += (y_pred.argmax(1) == y).sum().item()

    train_acc = correct / len(train_dataset)
    train_losses.append(train_loss / len(train_loader))
    train_accuracies.append(train_acc)

    # Evaluation
    model.eval()
    test_loss, test_correct = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            test_correct += (y_pred.argmax(1) == y).sum().item()

    test_acc = test_correct / len(test_dataset)
    test_losses.append(test_loss / len(test_loader))
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Test Loss: {test_losses[-1]:.4f} | Test Acc: {test_acc*100:.2f}%\n")

# ---- PLOT LOSS AND ACCURACY CURVES ----
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Test Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, label='Train Accuracy')
plt.plot(epochs_range, test_accuracies, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Test Accuracy')
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
# Ensure the model is in evaluation mode
model.eval()

# Get a batch of test images
X_batch, y_batch = next(iter(test_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Make predictions
with torch.no_grad():
    y_pred = model(X_batch)
    preds = torch.argmax(y_pred, dim=1)

# Move tensors back to CPU for visualization
X_batch = X_batch.cpu()
y_batch = y_batch.cpu()
preds = preds.cpu()

# Denormalize for visualization
def denormalize(img_tensor):
    """Undo normalization for plotting."""
    img_tensor = img_tensor.clone()
    img_tensor = img_tensor * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_tensor = img_tensor + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    return torch.clamp(img_tensor, 0, 1)

# Plot a few predictions
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img = denormalize(X_batch[i]).permute(1, 2, 0)
    true_label = y_batch[i].item()
    pred_label = preds[i].item()

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"True: {true_label} | Pred: {pred_label}",
                 color=("green" if true_label == pred_label else "red"))

plt.tight_layout()
plt.show()





# Partial fine tuning

In [ ]:
# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze later layers (e.g., inception4e, inception5a, inception5b)
for name, param in model.named_parameters():
    if any(layer in name for layer in ['inception4e', 'inception5a', 'inception5b', 'fc']):
        param.requires_grad = True


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

output_shape = len(set(y_train))

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=output_shape,
                    bias=True)).to(device)

In [ ]:
summary(model,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
x_train_tensor = torch.tensor(x_train_bal).unsqueeze(1).float() / 255.0  # (N, 1, 28, 28)
x_test_tensor = torch.tensor(x_test).unsqueeze(1).float() / 255.0
y_train_tensor = torch.tensor(y_train_bal).long()
y_test_tensor = torch.tensor(y_test).long()

# Convert grayscale to 3-channel (since GoogLeNet expects RGB)
x_train_tensor = x_train_tensor.repeat(1, 3, 1, 1)
x_test_tensor = x_test_tensor.repeat(1, 3, 1, 1)

# Apply transforms (resize + normalize)
train_data = []
for i in range(len(x_train_tensor)):
    img = transforms.ToPILImage()(x_train_tensor[i])
    img = manual_transforms(img)
    train_data.append(img)
x_train_tensor = torch.stack(train_data)

test_data = []
for i in range(len(x_test_tensor)):
    img = transforms.ToPILImage()(x_test_tensor[i])
    img = manual_transforms(img)
    test_data.append(img)
x_test_tensor = torch.stack(test_data)

# 2. Create DataLoaders
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Training loop
epochs = 7
for epoch in range(epochs):
    model.train()
    train_loss, correct = 0, 0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        X, y = X.to(device), y.to(device)

        # Forward pass
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # Backward + optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accuracy
        correct += (y_pred.argmax(1) == y).sum().item()

    train_acc = correct / len(train_dataset)
    print(f"Train loss: {train_loss/len(train_loader):.4f} | Train acc: {train_acc*100:.2f}%")

    # Evaluation loop
    model.eval()
    test_loss, test_correct = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            test_correct += (y_pred.argmax(1) == y).sum().item()
    test_acc = test_correct / len(test_dataset)
    print(f"Test loss: {test_loss/len(test_loader):.4f} | Test acc: {test_acc*100:.2f}%\n")


In [ ]:
# Lists to store results
train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []

# Training loop with metric tracking
epochs = 5
for epoch in range(epochs):
    model.train()
    train_loss, correct = 0, 0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        correct += (y_pred.argmax(1) == y).sum().item()

    train_acc = correct / len(train_dataset)
    train_losses.append(train_loss / len(train_loader))
    train_accuracies.append(train_acc)

    # Evaluation
    model.eval()
    test_loss, test_correct = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            test_correct += (y_pred.argmax(1) == y).sum().item()

    test_acc = test_correct / len(test_dataset)
    test_losses.append(test_loss / len(test_loader))
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Test Loss: {test_losses[-1]:.4f} | Test Acc: {test_acc*100:.2f}%\n")

# ---- PLOT LOSS AND ACCURACY CURVES ----
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Test Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, label='Train Accuracy')
plt.plot(epochs_range, test_accuracies, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Test Accuracy')
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
# Ensure the model is in evaluation mode
model.eval()

# Get a batch of test images
X_batch, y_batch = next(iter(test_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Make predictions
with torch.no_grad():
    y_pred = model(X_batch)
    preds = torch.argmax(y_pred, dim=1)

# Move tensors back to CPU for visualization
X_batch = X_batch.cpu()
y_batch = y_batch.cpu()
preds = preds.cpu()

# Denormalize for visualization
def denormalize(img_tensor):
    """Undo normalization for plotting."""
    img_tensor = img_tensor.clone()
    img_tensor = img_tensor * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_tensor = img_tensor + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    return torch.clamp(img_tensor, 0, 1)

# Plot a few predictions
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img = denormalize(X_batch[i]).permute(1, 2, 0)
    true_label = y_batch[i].item()
    pred_label = preds[i].item()

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"True: {true_label} | Pred: {pred_label}",
                 color=("green" if true_label == pred_label else "red"))

plt.tight_layout()
plt.show()


# Full Fine Tuning

In [ ]:
class MNISTDataset(Dataset):
    """Custom Dataset for MNIST grayscale -> 3-channel RGB (PIL-based)."""
    def __init__(self, data, labels, transform=None):
        """
        data: numpy array or torch tensor of shape (N, 28, 28)
        labels: array-like of integers (N,)
        transform: torchvision.transforms.Compose for preprocessing
        """
        if torch.is_tensor(data):
            data = data.cpu().numpy()
        if torch.is_tensor(labels):
            labels = labels.cpu().numpy()

        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.data[idx]
        label = int(self.labels[idx])

        # Convert grayscale to RGB using PIL (no deprecated 'mode' argument)
        img = Image.fromarray(img.astype('uint8')).convert("RGB")

        # Apply transforms if provided
        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
# 2) Define image transforms (GoogLeNet requires 224x224, normalized RGB)
# ----------------------------
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    torch.backends.cudnn.benchmark = True


In [ ]:
# 4) Create Dataset and DataLoaders
# ----------------------------
train_dataset = MNISTDataset(data=x_train_bal, labels=y_train_bal, transform=manual_transforms)
test_dataset  = MNISTDataset(data=x_test, labels=y_test, transform=manual_transforms)

BATCH_SIZE = 32
NUM_WORKERS = 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")
print(f"Batch size: {BATCH_SIZE} | num_workers: {NUM_WORKERS}")

In [ ]:
# 5) Model setup
# ----------------------------
model = torchvision.models.googlenet(
    weights=GoogLeNet_Weights.IMAGENET1K_V1 ,
).to(device)

# Replace final fully connected layer to match 10 MNIST classes
num_classes = 10
in_features = model.fc.in_features if hasattr(model, "fc") else 1024
model.fc = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=in_features, out_features=num_classes)
).to(device)

# Unfreeze all parameters for full fine-tuning
for p in model.parameters():
    p.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Trainable parameters: {trainable_params:,}")


In [ ]:
# 6) Loss function, optimizer, and scheduler
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
GRAD_CLIP_NORM = 5.0  # clip gradients to stabilize training

EPOCHS = 5
SAVE_PATH = "/content/best_mnist_googlenet_fullft_noaux.pth"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_acc = 0.0

In [ ]:
# 7) Training and validation loop
# ----------------------------
print(" Starting full fine-tuning ")

for epoch in range(1, EPOCHS + 1):
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    # --- TRAIN ---
    model.train()
    running_loss, running_correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    for X, y in loop:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()

        preds = logits.argmax(dim=1)
        running_correct += (preds == y).sum().item()
        running_loss += loss.item() * X.size(0)
        total += X.size(0)

        loop.set_postfix(loss=running_loss/total, acc=running_correct/total)

    avg_train_loss = running_loss / total
    train_acc = running_correct / total
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    # --- VALIDATE ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        val_loop = tqdm(test_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False)
        for Xv, yv in val_loop:
            Xv, yv = Xv.to(device, non_blocking=True), yv.to(device, non_blocking=True)
            out = model(Xv)
            loss_v = criterion(out, yv)
            val_loss += loss_v.item() * Xv.size(0)
            val_correct += (out.argmax(dim=1) == yv).sum().item()
            val_total += Xv.size(0)

    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

    scheduler.step()

    print(f"Epoch {epoch}/{EPOCHS} | Train loss: {avg_train_loss:.4f} | Train acc: {train_acc*100:.2f}% "
          f"| Val loss: {avg_val_loss:.4f} | Val acc: {val_acc*100:.2f}%")

    # Save best model checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
        }, SAVE_PATH)
        print(f"Saved new best model (val_acc={val_acc*100:.2f}%) → {SAVE_PATH}")

print("Training complete!")


In [ ]:
# 8) Plot training and validation metrics
# ----------------------------
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Val Loss")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.title("Loss Curves")
plt.legend(); plt.grid(True)

plt.subplot(1,2,2)
plt.plot(epochs_range, train_accs, label="Train Acc")
plt.plot(epochs_range, val_accs, label="Val Acc")
plt.xlabel("Epochs"); plt.ylabel("Accuracy"); plt.title("Accuracy Curves")
plt.legend(); plt.grid(True)
plt.show()


In [ ]:
# 9) Visualization: show some predictions
# ----------------------------
def denormalize(img_tensor):
    """Undo ImageNet normalization for visualization."""
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    mean= torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    img = img_tensor.clone().cpu()
    img = img * std + mean
    return torch.clamp(img, 0, 1)

model.eval()
X_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    X_gpu = X_batch.to(device)
    preds = model(X_gpu).argmax(dim=1).cpu()

fig, axes = plt.subplots(3,5, figsize=(12,7))
axes = axes.flatten()
for i, ax in enumerate(axes):
    img = denormalize(X_batch[i]).permute(1,2,0)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"T:{int(y_batch[i])} P:{int(preds[i])}",
                 color=("green" if int(y_batch[i])==int(preds[i]) else "red"))
plt.tight_layout()
plt.show()